In [4]:
!pip install xgboost
!pip install seaborn

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor, MultiOutputClassifier
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb
from xgboost import plot_importance
import pickle
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

### Loading Data

In [2]:
df = pd.read_csv("cocktail_dataset_all.csv")
print(f"Dataset shape: {df.shape}")

Dataset shape: (4603, 244)


### Input/Output

INPUT: All ingredient percentages
OUTPUT: 3 ratings + 9 categories

In [3]:
ingredient_cols = [col for col in df.columns if '_pct' in col]
X = df[ingredient_cols]
print(f"Input features: {len(ingredient_cols)} ingredient percentages")

rating_outputs = ['strength_rating', 'taste_rating']
# category_outputs = ['bittersweet', 'citrus', 'creamy', 'sweet', 'floral', 'fruity', 'herbal', 'savoury', 'spicy']

# print(f"Output targets: {len(rating_outputs)} ratings + {len(category_outputs)} categories")
print(f"Output targets: {len(rating_outputs)} ratings")

Input features: 240 ingredient percentages
Output targets: 2 ratings


### Splitting Data

In [4]:
"""Split data into training and testing sets"""
X_train, X_test, y_train, y_test = train_test_split(
    X, df[rating_outputs], 
    test_size=0.2, random_state=42
)

print(f"Training: {X_train.shape[0]} recipes")
print(f"Testing: {X_test.shape[0]} recipes")

Training: 3682 recipes
Testing: 921 recipes


In [5]:
rating_model = MultiOutputRegressor(
    xgb.XGBRegressor(
        # ⚡ MAJOR UPGRADES:
        n_estimators=300,           # ↑ FROM 300 (2.7x more trees)
        learning_rate=0.05,         # ↓ FROM 0.1 (slower, more precise)
        max_depth=7,                # ↑ FROM 5 (deeper for complex interactions)
        
        # 🛡️ NEW REGULARIZATION (prevents overfitting):
        min_child_weight=4,         # Require more samples in leaf nodes
        subsample=0.8,              # Use 80% of data randomly each iteration
        colsample_bytree=0.7,       # Use 70% of features randomly each tree
        reg_alpha=0.3,              # L1 regularization (feature selection)
        reg_lambda=1.2,             # L2 regularization (smoothing)
        gamma=0.1,                  # Minimum loss reduction for splits
        
        random_state=42,
        n_jobs=-1,
        objective='reg:squarederror'
    )
)
rating_model.fit(X_train, y_train[rating_outputs])

,estimator,"XGBRegressor(...ree=None, ...)"
,n_jobs,None
,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.7
,device,None
,early_stopping_rounds,None


In [6]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

prediction = rating_model.predict(X_test)
for i, col in enumerate(rating_outputs):
    preds = prediction[:, i]
    actual = y_test[col].values
    
    # Original metrics
    rmse = np.sqrt(mean_squared_error(actual, preds))
    r2 = r2_score(actual, preds)
    mae = mean_absolute_error(actual, preds)

    # Tolerance-based metrics
    abs_diff = np.abs(preds - actual)
    
    # Calculate various accuracy metrics
    exact_match = np.mean(abs_diff == 0) * 100
    within_one = np.mean(abs_diff <= 1) * 100
    within_half = np.mean(abs_diff <= 0.5) * 100
    
    # Error distribution
    mean_abs_error = np.mean(abs_diff)
    std_abs_error = np.std(abs_diff)
    
    print(f"\n{col}:")
    print(f"  RMSE:           {rmse:.3f}")
    print(f"  MAE:            {mae:.3f}")
    print(f"  R²:             {r2:.3f}")
    print(f"  Exact match:    {exact_match:.2f}%")
    print(f"  Within ±0.5:    {within_half:.2f}%")
    print(f"  Within ±1:      {within_one:.2f}%")
    print(f"  Mean abs error: {mean_abs_error:.3f} ± {std_abs_error:.3f}")


strength_rating:
  RMSE:           1.055
  MAE:            0.792
  R²:             0.477
  Exact match:    0.00%
  Within ±0.5:    43.43%
  Within ±1:      71.23%
  Mean abs error: 0.792 ± 0.697

taste_rating:
  RMSE:           0.852
  MAE:            0.651
  R²:             0.351
  Exact match:    0.00%
  Within ±0.5:    49.51%
  Within ±1:      79.48%
  Mean abs error: 0.651 ± 0.550


In [8]:
with open('./taste_rating_model.pkl', 'wb') as f:
    pickle.dump(rating_model, f)

In [9]:
df = pd.read_csv("cocktail_dataset_2700.csv")
print(f"Dataset shape: {df.shape}")

Dataset shape: (2700, 254)


In [10]:
ingredient_cols = [col for col in df.columns if '_pct' in col]
X = df[ingredient_cols]
print(f"Input features: {len(ingredient_cols)} ingredient percentages")

category_outputs = ['bittersweet', 'citrus', 'creamy', 'sweet', 'floral', 'fruity', 'herbal', 'savoury', 'spicy', 'nutty']

print(f"Output targets: {len(category_outputs)} ratings")

Input features: 240 ingredient percentages
Output targets: 10 ratings


In [11]:
"""Split data into training and testing sets"""
X_train, X_test, y_train, y_test = train_test_split(
    X, df[category_outputs], 
    test_size=0.2, random_state=42
)

print(f"Training: {X_train.shape[0]} recipes")
print(f"Testing: {X_test.shape[0]} recipes")

Training: 2160 recipes
Testing: 540 recipes


In [12]:
# rating_model = MultiOutputRegressor(
#     xgb.XGBRegressor(
#         # ⚡ MAJOR UPGRADES:
#         n_estimators=300,           # ↑ FROM 300 (2.7x more trees)
#         learning_rate=0.05,         # ↓ FROM 0.1 (slower, more precise)
#         max_depth=7,                # ↑ FROM 5 (deeper for complex interactions)
        
#         # 🛡️ NEW REGULARIZATION (prevents overfitting):
#         min_child_weight=4,         # Require more samples in leaf nodes
#         subsample=0.8,              # Use 80% of data randomly each iteration
#         colsample_bytree=0.7,       # Use 70% of features randomly each tree
#         reg_alpha=0.3,              # L1 regularization (feature selection)
#         reg_lambda=1.2,             # L2 regularization (smoothing)
#         gamma=0.1,                  # Minimum loss reduction for splits
        
#         random_state=42,
#         n_jobs=-1,
#         objective='reg:squarederror'
#     )
# )
# rating_model.fit(X_train, y_train[rating_outputs])

category_model = MultiOutputClassifier(
    xgb.XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=4,
        random_state=42,
        n_jobs=-1
    )
)
category_model.fit(X_train, y_train[category_outputs])
print("✓ Category model trained")

✓ Category model trained


In [13]:
rating_preds = rating_model.predict(X_test)
rating_preds_df = pd.DataFrame(rating_preds, columns=rating_outputs)

# Predict categories (probabilities)
category_probs = category_model.predict_proba(X_test)
category_preds = category_model.predict(X_test)
category_preds_df = pd.DataFrame(category_preds, columns=category_outputs)

In [14]:
for i, col in enumerate(category_outputs):
    accuracy = accuracy_score(y_test[col], category_preds[:, i])
    print(f"{col:<10} Accuracy: {accuracy:.3f}")

# Overall category accuracy
avg_accuracy = np.mean([
    accuracy_score(y_test[col], category_preds[:, i]) 
    for i, col in enumerate(category_outputs)
])
print(f"\nAverage category accuracy: {avg_accuracy:.3f}")

bittersweet Accuracy: 0.915
citrus     Accuracy: 0.822
creamy     Accuracy: 0.987
sweet      Accuracy: 0.972
floral     Accuracy: 0.963
fruity     Accuracy: 0.857
herbal     Accuracy: 0.874
savoury    Accuracy: 0.969
spicy      Accuracy: 0.963
nutty      Accuracy: 0.991

Average category accuracy: 0.931


In [15]:
with open('./keywords_model.pkl', 'wb') as f:
    pickle.dump(category_model, f)